# BC pipeline walkthrough

Reproduces the BC + PPO warm-start pipeline of the dissertation on a
scaled-down budget:

1. Inspect the shipped BC teacher dataset at `data/BC/training/`
   (313 394 RAISocketAI demonstrations across 6 NPZ chunks).
2. Train a BC agent for 3 epochs (smoke budget; the published
   `UECD-BC` ran 30 epochs).
3. Evaluate the just-trained BC agent against `RandomBiasedAI`.
4. Show how to chain into PPO fine-tuning (the `UECD-BC-PPO` step
   from the thesis).

BC training is GPU-friendly but works on CPU; the smoke budget here
is ~3 minutes on CPU. The full thesis BC run takes ~30 minutes on
GPU; the BC -> PPO chain to reach 96% pool WR adds another 100 M
PPO steps (~10 hours on GPU).

**Prereqs**: `bash setup/local.sh` once. Run from the repo root.

## 1. Inspect the BC training data

`data/BC/training/` ships 6 NPZ chunks: RAISocketAI playing 100 games
against each of three opponents (itself, CoacAI, Mayari) on
`basesWorkers16x16A`. Each chunk records per-cell observations + gridnet
actions + sparse rewards.

In [ ]:
import glob

import numpy as np

from microrts_agent.paths import PROJECT_ROOT

bc_dir = PROJECT_ROOT / "data" / "BC" / "training"
chunks = sorted(glob.glob(str(bc_dir / "bc_chunk_*.npz")))
print(f"Chunks found: {len(chunks)}")

total_transitions = 0
for cp in chunks:
    d = np.load(cp)
    n = len(d["obs"])
    total_transitions += n
    print(f"  {cp.split('/')[-1]:35s} obs={d['obs'].shape} actions={d['actions'].shape} n={n:,}")
print(f"\nTotal transitions: {total_transitions:,}")

## 2. Run BC training (smoke budget)

3 epochs, GridNet architecture, batch size 128. ~3 min on CPU.
The output ends up at `outputs/runs/bc-smoke_s1/{agent.pt,config.json}`.

In [ ]:
import subprocess
import sys

bc_run_dir = PROJECT_ROOT / "outputs" / "runs" / "bc-smoke_s1"

cmd = [
    sys.executable,
    "-m",
    "microrts_agent",
    "bc",
    "train",
    "--data",
    *chunks,
    "--architecture",
    "gridnet",
    "--epochs",
    "3",
    "--batch-size",
    "128",
    "--lr",
    "1e-4",
    "--seed",
    "1",
    "--output",
    str(bc_run_dir),
]
print("Running:", cmd[0], cmd[1], cmd[2], cmd[3], cmd[4], "--data", "<6 NPZ files>", *cmd[7:])
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True, timeout=900)
print("--- last 1500 chars of stdout ---")
print(result.stdout[-1500:])
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr[-1500:])

## 3. Evaluate the just-trained BC agent

BC at 3 epochs is severely under-trained: this is a smoke check that
the pipeline produced a loadable agent, not a benchmark. For the
real 78% BC-only baseline, see
[`data/BC/baseline/`](../data/BC/baseline/) (UECD-BC, 30 epochs over
the full dataset).

In [ ]:
eval_cmd = [
    sys.executable,
    "-m",
    "microrts_agent",
    "evaluate",
    "--agent",
    str(bc_run_dir),
    "--opponent",
    "RandomBiasedAI",
    "--maps",
    "maps/open_competition/basesWorkers16x16A.xml",
    "--nb_games",
    "5",
    "--max-steps",
    "2000",
]
result = subprocess.run(
    eval_cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True, timeout=300
)
print(result.stdout[-1500:])

## 4. Chaining into PPO (sketch)

The dissertation's BC+PPO recipe takes the just-trained BC agent
(its `agent.pt`) and resumes PPO training from it, optionally with a
KL teacher penalty against the frozen BC policy.

Don't run this cell unless you have a GPU + time: the full
`UECD-BC-PPO` run is 100 M PPO steps on top of BC. The snippet below
is for reference; the SLURM driver is at
[`experiments/BC/train_UECD-BC-PPO.slurm`](../experiments/BC/train_UECD-BC-PPO.slurm).

In [ ]:
ppo_cmd_sketch = [
    "microrts-agent",
    "train",
    "--exp-name",
    "bc-smoke-ppo_s1",
    "--architecture",
    "gridnet",
    "--map",
    "maps/open_competition/basesWorkers16x16A.xml",
    "--total-timesteps",
    "1000000",  # 1 M for a quick demo (10 min CPU)
    "--num-bot-envs",
    "8",
    "--seed",
    "1",
    "--load-model",
    str(bc_run_dir / "agent.pt"),  # warm-start from BC
]
print("To run BC -> PPO fine-tune:")
print("  " + " \\\n  ".join(ppo_cmd_sketch))
print()
print("(Not executed in this notebook; uncomment subprocess.run below if you want to try.)")
# subprocess.run(ppo_cmd_sketch, cwd=str(PROJECT_ROOT), timeout=900)

## Next steps

- Full BC reproduction: run
  [`experiments/BC/train_UECD-BC.slurm`](../experiments/BC/train_UECD-BC.slurm)
  on a GPU node to reproduce `UECD-BC` (78% pool WR after 30 epochs).
- Full BC+PPO reproduction:
  [`experiments/BC/train_UECD-BC-PPO.slurm`](../experiments/BC/train_UECD-BC-PPO.slurm)
  lifts the 78% baseline to ~96% with 100 M PPO steps.
- Inspect the published BC baseline numbers under
  [`data/BC/baseline/results.csv`](../data/BC/baseline/results.csv)
  (per-opponent breakdown, 1000-game eval per cell).
- Regenerate the teacher dataset with different opponents:
  `microrts-agent bc generate --bot RAISocketAI --opponents <list>`
  (see `microrts_agent/bc/generate.py --help`).